# Create eval data

Run the agent on every usage scenario in `questions.csv` and save the answers, tool calls
and cost to `evals_run_<date>.json`, ready for review in the next step.

Run this notebook from the `evals/` folder. Requires `OPENAI_API_KEY` in the project's `.env`.

## Loading questions

In [1]:
import pandas as pd

df_questions = pd.read_csv('questions.csv')
df_questions.head()

,question,group,type
0,What processor does the Alexy G26 Pro use?,product_specs,direct
1,Does the G26 Lite have a fingerprint reader?,product_specs,wrong_terminology
2,Which of your phones has the biggest battery a...,product_specs,multi_source
3,Can I take the G26 Lite swimming?,product_specs,conflicting_sources
4,Does the G26 support wireless charging?,product_specs,not_in_docs


## Setting up the agent

The same way as in the app and the tests:

In [2]:
import sys
sys.path.insert(0, '..')  # project root, so `tests.utils` can be imported

from grigotel_ai_assistant.agent import MODEL, SYSTEM_PROMPT, GrigoTelAgent
from grigotel_ai_assistant.session_cost import calculate_cost
from grigotel_ai_assistant.tools import create_tools
from tests.utils import collect_tools

tools = create_tools()

Logfire project URL: https://logfire-eu.pydantic.dev/nima-dx/grigotel-ai-assistant

A wrapper to run the agent on one question. It's the same `chat_stream` the Streamlit app
uses, without the UI. Our agent is synchronous, so no `await` is needed (the lesson's
PydanticAI agent is async).

In [3]:
def run_agent(agent, user_prompt):
    final = None
    for event in agent.chat_stream(user_prompt):
        if event['type'] == 'final':
            final = event
    return final

## Extracting cost and tool calls

Test on the first question:

In [4]:
row = df_questions.iloc[0]

agent = GrigoTelAgent(tools)
result = run_agent(agent, row.question)
result['response']

18:57:46.284 Responses API with 'gpt-4o-mini' [LLM]
18:57:47.881 streaming response from 'gpt-4o-mini' took 0.63s [LLM]
18:57:47.887 Responses API with 'gpt-4o-mini' [LLM]
18:57:50.244 streaming response from 'gpt-4o-mini' took 1.79s [LLM]


AssistantResponse(answer='The Alexy G26 Pro uses the Helios G1 Neural Chip, which features a 4nm architecture.', answer_type='product_specs', confidence=1.0, found_in_sources=True, sources=['Alexy_G26_Pro_GrigoTel_Product_Specifications.pdf'], follow_up_questions=['What are the other specs of the Alexy G26 Pro?', 'How does it compare to the Alexy G26?', 'What is the price of the Alexy G26 Pro?'])

`agent.last_usage` holds the tokens of the last customer message (summed over all its LLM calls):

In [5]:
usage = agent.last_usage

cost = calculate_cost(
    model_name=MODEL,
    input_tokens=usage['input_tokens'],
    output_tokens=usage['output_tokens'],
)
cost

0.00061275

Which tools did the agent call, with which arguments?

In [6]:
tools_list = collect_tools(agent.messages)
tools_list

[{'name': 'search_docs',
  'arguments': {'query': 'processor', 'model': 'Alexy G26 Pro'}}]

## Running all scenarios

Each question gets a fresh agent (a new conversation), **except `follow_up` questions**:
they only make sense after the previous question (e.g. *"And how much does it cost?"*),
so they continue the previous row's conversation. `collect_tools` only looks at the
messages of the current question, so the previous question's tool calls aren't counted.

In [7]:
agent = None


def run_eval_round(row):
    global agent

    previous_question = None
    if row.type == 'follow_up' and agent is not None:
        previous_question = agent.messages[0]['content']
    else:
        agent = GrigoTelAgent(tools)  # new conversation

    history_length = len(agent.messages)
    result = run_agent(agent, row.question)

    usage = agent.last_usage
    cost = calculate_cost(
        model_name=MODEL,
        input_tokens=usage['input_tokens'],
        output_tokens=usage['output_tokens'],
    )

    tools_list = collect_tools(agent.messages[history_length:])

    output = {
        'input': row.to_dict(),
        'previous_question': previous_question,
        'system_prompt': SYSTEM_PROMPT,
        'response': result['response'].model_dump(),
        'tools': tools_list,
        'order': result['order'],
        'usage': usage,
        'cost': cost,
    }

    return output

We use `tqdm` for a progress bar and `try/except`, so one failure (a rate limit, too many
tool rounds, ...) doesn't stop the whole batch:

In [8]:
import traceback
from tqdm.auto import tqdm

outputs = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    try:
        output = run_eval_round(row)
        outputs.append(output)
    except Exception as e:
        print(f"Cannot process row {row.to_dict()}: {e}")
        traceback.print_exc()

  0%|          | 0/20 [00:00<?, ?it/s]

18:58:46.349 Responses API with 'gpt-4o-mini' [LLM]
18:58:47.967 streaming response from 'gpt-4o-mini' took 1.02s [LLM]
18:58:47.980 Responses API with 'gpt-4o-mini' [LLM]
18:58:50.328 streaming response from 'gpt-4o-mini' took 1.51s [LLM]
18:58:50.335 Responses API with 'gpt-4o-mini' [LLM]
18:58:52.371 streaming response from 'gpt-4o-mini' took 1.03s [LLM]
18:58:52.383 Responses API with 'gpt-4o-mini' [LLM]
18:58:54.833 streaming response from 'gpt-4o-mini' took 1.75s [LLM]
18:58:54.841 Responses API with 'gpt-4o-mini' [LLM]
18:58:58.417 streaming response from 'gpt-4o-mini' took 3.02s [LLM]
18:58:58.446 Responses API with 'gpt-4o-mini' [LLM]
18:59:03.240 streaming response from 'gpt-4o-mini' took 4.11s [LLM]
18:59:03.248 Responses API with 'gpt-4o-mini' [LLM]
18:59:04.810 streaming response from 'gpt-4o-mini' took 0.87s [LLM]
18:59:04.823 Responses API with 'gpt-4o-mini' [LLM]
18:59:07.832 streaming response from 'gpt-4o-mini' took 2.40s [LLM]
18:59:07.840 Responses API with 'gpt-4o-

## Saving the results

The file name gets the date of the run, e.g. `evals_run_2026_11_09.json`. If you run it
more than once on the same day, a number is added (`evals_run_2026_11_09_2.json`) so
earlier runs are not overwritten.

In [9]:
import json
from datetime import date
from pathlib import Path

base_name = f'evals_run_{date.today():%Y_%m_%d}'
output_path = Path(f'{base_name}.json')
run_number = 1
while output_path.exists():
    run_number += 1
    output_path = Path(f'{base_name}_{run_number}.json')

with open(output_path, 'wt') as f_out:
    json.dump(outputs, f_out, indent=2, ensure_ascii=False)

print(f'saved {len(outputs)} of {len(df_questions)} results to {output_path}')

saved 20 of 20 results to evals_run_2026_11_10.json


Check the total cost:

In [10]:
total_cost = sum([o['cost'] for o in outputs])
print(f'total cost: {total_cost:0.3f}')

total cost: 0.018
